In [1]:
import pandas as pd
import numpy as np
from helpers import get_feature_importance
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from joblib import dump, load

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error as mse, mean_absolute_error as mae, mean_absolute_percentage_error as mape, r2_score
from category_encoders.target_encoder import TargetEncoder
seed = 42

## Тюнинг гиперпараметров моделей

Для повышения качества моделей был выполнен поэтапный тюнинг гиперпараметров.

На первом этапе использовался RandomizedSearchCV с широкими диапазонами значений гиперпараметров. Это позволило эффективно исследовать пространство параметров и определить области, в которых модели показывают наилучшие результаты, без чрезмерных вычислительных затрат.

На втором этапе диапазоны значений были сужены вокруг найденных оптимальных областей, после чего применён GridSearchCV. Такой подход позволил более точно подобрать оптимальные значения гиперпараметров.

Таким образом, комбинированная стратегия RandomizedSearchCV → GridSearchCV позволила сначала глобально исследовать пространство параметров, а затем выполнить точную настройку моделей.


In [3]:
df = pd.read_csv('diamonds.csv')
df.drop(columns=['Unnamed: 0', 'x', 'y', 'z'], inplace=True)
result_cross_val_baseline = load('result_cross_val_baseline.joblib')
result_test_baseline = load('result_test_baseline.joblib')

In [4]:
# ====== 1. Разделение на X и y ======
X = df.drop('price', axis=1)
y = df['price']

# ====== 2. Определение типов признаков ======
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()
te_cals = ['cut', 'color', 'clarity']

# ====== 3. Препроцессинг ======
num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

te_pipeline = Pipeline([
    ('target_encoder', TargetEncoder()),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
    ('te', te_pipeline, te_cals)
])

# ====== 4. Разбиение ======
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=seed
)

# ====== 5. Модели ======
models = {
    'XGBoost': XGBRegressor(
        random_state=seed
    )
}

In [6]:
# ====== ТЮНИНГ ГИПЕРПАРАМЕТРОВ ======================== 

param_rs_grids = {
    'XGBoost': {
        'model__n_estimators': [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750 , 800, 850, 900, 950, 1000],
        'model__max_depth': [2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30],
        'model__learning_rate': [0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2],
        'model__subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1],
        'model__colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1],
        'model__min_child_weight': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    }
}

cv = KFold(n_splits=5, shuffle=True, random_state=seed)

best_models_rs = {}
best_params_rs = {}

for name, model in models.items():
    print(f"\nTuning {name}...")
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    random_search = RandomizedSearchCV(
        estimator = pipeline,
        param_distributions = param_rs_grids[name],
        n_iter = 100,
        cv=cv,
        scoring='neg_mean_absolute_percentage_error',   # главная метрика 
        n_jobs=-1,
        verbose=1
    )
    
    random_search.fit(X_train, y_train)
    
    best_models_rs[name] = random_search.best_estimator_
    best_params_rs[name] = random_search.best_params_

# Таблица лучших параметров
best_params_rs_df = pd.DataFrame(best_params_rs)


Tuning XGBoost...
Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [16]:
results_tuned_rs = {}

for name, best_model in best_models_rs.items():
    
    y_pred = best_model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mse_score = mse(y_test, y_pred)
    mae_score = mae(y_test, y_pred)
    mape_score = mape(y_test, y_pred)
    rmse_score = np.sqrt(mse(y_test, y_pred))
    
    results_tuned_rs[name] = [r2, mse_score, mae_score, mape_score, rmse_score]

result_test_tuned_rs_df = pd.DataFrame(
    results_tuned_rs,
    index=['r2', 'mse', 'mae', 'mape', 'rsme'] ).T

print("\nBaseline CV results:")
display(result_cross_val_baseline)

print("\nBest hyperparameters:")
display(best_params_rs_df.fillna('-'))

print("\nTest results baseline:")
display(result_test_baseline)

print("\nTest results after tuning:")
display(result_test_tuned_rs_df)

print("\nCompare baseline and tuned:")
display(result_test_tuned_rs_df-result_test_baseline)


Baseline CV results:


,r2,mse,mae,mape,rsme
XGBoost,0.9781 ± 0.0008,348961.5188 ± 11867.2972,300.5996 ± 5.1821,0.0841 ± 0.0012,590.6448 ± 10.0101



Best hyperparameters:


,XGBoost
model__subsample,0.80
model__n_estimators,800.00
model__min_child_weight,3.00
model__max_depth,9.00
model__learning_rate,0.03
model__colsample_bytree,1.00



Test results baseline:


,r2,mse,mae,mape,rsme
XGBoost,0.978765,337576.375,295.337585,0.083433,581.013231



Test results after tuning:


,r2,mse,mae,mape,rsme
XGBoost,0.980208,314626.625,272.937134,0.073624,560.915881



Compare baseline and tuned:


,r2,mse,mae,mape,rsme
XGBoost,0.001444,-22949.75,-22.400452,-0.009809,-20.097351


In [18]:
# ====== ТЮНИНГ ГИПЕРПАРАМЕТРОВ GridSearchCV =============

param_gridsearch = {
    'XGBoost': {
        'model__n_estimators': [750, 800, 850],
        'model__max_depth': [8, 9, 10],
        'model__learning_rate': [0.02, 0.03, 0.04],
        #'model__subsample': [0.75, 0.8, 0.9],
        #'model__colsample_bytree': [0.9, 1],
        #'model__min_child_weight': [2, 3, 4]
    }
}

cv = KFold(n_splits=5, shuffle=True, random_state=seed) 

best_models_gs = {}
best_params_gs = {}

for name, model in models.items():
    print(f"\nTuning {name}...")
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    grid_search = GridSearchCV(
        estimator = pipeline,
        param_grid = param_gridsearch[name],
        cv=cv,
        scoring='neg_mean_absolute_percentage_error',   # главная метрика
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    best_models_gs[name] = grid_search.best_estimator_
    best_params_gs[name] = grid_search.best_params_

# Таблица лучших параметров
best_params_gs_df = pd.DataFrame(best_params_gs)


Tuning XGBoost...
Fitting 5 folds for each of 27 candidates, totalling 135 fits


In [22]:
results_tuned_gs = {}

for name, best_model in best_models_gs.items():
    
    y_pred = best_model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mse_score = mse(y_test, y_pred)
    mae_score = mae(y_test, y_pred)
    mape_score = mape(y_test, y_pred)
    rmse_score = np.sqrt(mse(y_test, y_pred))
    
    results_tuned_gs[name] = [r2, mse_score, mae_score, mape_score, rmse_score]

result_test_tuned_gs_df = pd.DataFrame(
    results_tuned_gs,
    index=['r2', 'mse', 'mae', 'mape', 'rsme'] ).T

print("\nBaseline CV results:")
display(result_cross_val_baseline)

print("\nBest hyperparameters:")
display(best_params_gs_df.fillna('-'))

print("\nTest results after tuning:")
display(result_test_tuned_gs_df)

print("\nTest results baseline:")
display(result_test_baseline)

print("\nCompare baseline and tuned:")
display(result_test_tuned_gs_df-result_test_baseline)


Baseline CV results:


,r2,mse,mae,mape,rsme
XGBoost,0.9781 ± 0.0008,348961.5188 ± 11867.2972,300.5996 ± 5.1821,0.0841 ± 0.0012,590.6448 ± 10.0101



Best hyperparameters:


,XGBoost
model__learning_rate,0.02
model__max_depth,9.00
model__n_estimators,800.00



Test results after tuning:


,r2,mse,mae,mape,rsme
XGBoost,0.981414,295465.375,270.09787,0.073401,543.567268



Test results baseline:


,r2,mse,mae,mape,rsme
XGBoost,0.978765,337576.375,295.337585,0.083433,581.013231



Compare baseline and tuned:


,r2,mse,mae,mape,rsme
XGBoost,0.002649,-42111.0,-25.239716,-0.010032,-37.445963


## Итог
Тюнинг гиперпараметров дал небольшой "буст" по всем метрикам: 

r2: +0.3%

mse: -42111

mae: -25.2

mape: -1%

rmse: -37.45